In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np

In [3]:
np.random.seed(42)

In [4]:
income=np.random.normal(5000,2000,1000)

In [5]:
sq_ft=np.random.uniform(800,4000,1000)

In [6]:
zip_codes = [f"ZIP_{np.random.randint(10000, 10500)}" for _ in range(1000)]

In [7]:
random_noise=np.random.uniform(0,100,1000)

In [8]:
house_price = (sq_ft * 150) + (income * 2) + np.random.normal(0, 50000, 1000)

In [9]:
sq_meters = sq_ft / 10.764

In [10]:
import pandas as pd

In [11]:
data = pd.DataFrame({
    'Income': income,
    'Square_Footage': sq_ft,
    'Square_Meters': sq_meters,
    'ZipCode': zip_codes,
    'Random_Noise': random_noise,
    'House_Price': house_price
})

In [12]:
data

,Income,Square_Footage,Square_Meters,ZipCode,Random_Noise,House_Price
0,5993.428306,1335.944263,124.112250,ZIP_10189,85.422191,213544.971529
1,4723.471398,1134.617089,105.408500,ZIP_10135,77.308904,156259.539131
2,6295.377076,2836.576799,263.524415,ZIP_10042,30.999994,384252.410513
3,8046.059713,3060.722325,284.348042,ZIP_10403,23.203983,486264.774226
4,4531.693251,901.075663,83.711972,ZIP_10379,45.823465,176450.049781
...,...,...,...,...,...,...
995,4437.799414,1698.186290,157.765356,ZIP_10449,42.354792,294148.425971
996,8595.373054,1462.376251,135.858069,ZIP_10417,79.357065,218965.925365
997,6281.685723,2453.032411,227.892272,ZIP_10246,3.177848,395289.723685
998,3857.642020,817.618557,75.958617,ZIP_10007,78.205077,54611.853869


In [13]:
x=data.drop(columns=['House_Price'])

In [14]:
y=data['House_Price']

In [15]:
num_cols=x.select_dtypes(include=[np.number]).columns.tolist()

In [16]:
cat_cols=x.select_dtypes(exclude=[np.number]).columns.tolist()

In [17]:
#VIF for multi-collinearity

In [18]:
import statsmodels.api as sm

In [19]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [20]:
if num_cols:
    X_num = x[num_cols].dropna()
    X_with_constant=sm.add_constant(X_num)
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X_with_constant.columns
    print(vif_data)
    vifs = []
    for i in range(X_with_constant.shape[1]):
        try:
            vif_val = variance_inflation_factor(X_with_constant.values, i)
        except:
            vif_val = float('inf') 
        vifs.append(vif_val)
        
    vif_data["VIF_Score"] = vifs
    vif_data = vif_data[vif_data["Feature"] != "const"].sort_values(by="VIF_Score", ascending=False)
    
    print("--- Variance Inflation Factor (VIF) Report ---")
    for _, row in vif_data.iterrows():
        feature = row['Feature']
        score = row['VIF_Score']
        
        if score > 10.0 or score == float('inf'):
            print(f"[FATAL] {feature}: {score:.2f} (Perfect collinearity. MUST DROP!)")
        elif score > 5.0:
            print(f"[WARNING] {feature}: {score:.2f} (High collinearity. Investigate.)")
        else:
            print(f"[SAFE] {feature}: {score:.2f}")
else:
    print("No numerical columns found to calculate VIF.")




          Feature
0           const
1          Income
2  Square_Footage
3   Square_Meters
4    Random_Noise
--- Variance Inflation Factor (VIF) Report ---
[FATAL] Square_Footage: inf (Perfect collinearity. MUST DROP!)
[FATAL] Square_Meters: inf (Perfect collinearity. MUST DROP!)
[SAFE] Random_Noise: 1.00
[SAFE] Income: 1.00


In [21]:
vif_data

,Feature,VIF_Score
2,Square_Footage,inf
3,Square_Meters,inf
4,Random_Noise,1.002492
1,Income,1.001504


In [22]:
#Tree and ensemble EDA

In [23]:
if cat_cols:
    for col in cat_cols:
        unique_count = x[col].nunique()
        if unique_count > 15:
            print(f"[WARNING] {col}: {unique_count} unique values. (Use Target Encoding, NOT One-Hot!)")
        else:
            print(f"[SAFE] {col}: {unique_count} unique values. (Safe for One-Hot Encoding)")
else:
    print("No categorical columns detected.")

[WARNING] ZipCode: 434 unique values. (Use Target Encoding, NOT One-Hot!)


In [24]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_regression

In [25]:
X_mi = x.copy()

for col in cat_cols:
    X_mi[col] = LabelEncoder().fit_transform(X_mi[col].astype(str))

X_mi = X_mi.fillna(X_mi.median())

mi_scores = mutual_info_regression(X_mi, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X_mi.columns).sort_values(ascending=False)

for feature, score in mi_series.items():
    if score <= 0.01:
        print(f"[FLAGGED] {feature}: {score:.4f} (Pure noise. Consider dropping.)")
    else:
        print(f"[SIGNAL] {feature}: {score:.4f}")

[SIGNAL] Square_Footage: 1.0453
[SIGNAL] Square_Meters: 1.0453
[FLAGGED] Income: 0.0000 (Pure noise. Consider dropping.)
[FLAGGED] ZipCode: 0.0000 (Pure noise. Consider dropping.)
[FLAGGED] Random_Noise: 0.0000 (Pure noise. Consider dropping.)


In [28]:
#For neural networks

In [27]:
for col in num_cols:
    mean_val = x[col].mean()
    std_val = x[col].std()
    
    if abs(mean_val) > 0.5 or std_val > 2.0:
        print(f"[FATAL] {col} | Mean: {mean_val:.2f} | Std: {std_val:.2f}")
        print(f"        -> Will cause gradient saturation. Scaling mandatory.")
    else:
        print(f"[SAFE] {col} | Mean: {mean_val:.2f} | Std: {std_val:.2f}")

[FATAL] Income | Mean: 5038.66 | Std: 1958.43
        -> Will cause gradient saturation. Scaling mandatory.
[FATAL] Square_Footage | Mean: 2411.68 | Std: 922.74
        -> Will cause gradient saturation. Scaling mandatory.
[FATAL] Square_Meters | Mean: 224.05 | Std: 85.72
        -> Will cause gradient saturation. Scaling mandatory.
[FATAL] Random_Noise | Mean: 51.43 | Std: 28.36
        -> Will cause gradient saturation. Scaling mandatory.
